![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Labs

----

# Receiving the OpenLineage events

The data now moves along the supply chains.  Apache Airflow runs the DAGs in [`exchange-quickstart/coco-dags`](../../exchange-quickstart/coco-dags/README.md): the `coco_feed_*` DAGs copy each change from the systems into the digital products' tables in `coco_data_hub`, and the `coco_subscriptions_*` DAGs apply what the products' subscriptions deliver to the subscribing systems.  Airflow reports what each task reads and writes as [OpenLineage](https://egeria-project.org/features/lineage-management/overview/#the-open-lineage-standard) events.

In this workbook, [Peter Profile](https://egeria-project.org/practices/coco-pharmaceuticals/personas/peter-profile/) connects Egeria to those events.  They travel from Airflow, through the OpenLineage proxy in `shared-infra`, to the Apache Kafka topic `openlineage.events`.  Peter registers that topic as a catalog target of Egeria's **`OpenLineageKafkaListener`** integration connector, which passes each event to the **`OpenLineageCataloguer`**.  The cataloguer records each DAG task as a process and links it with `DataFlow` lineage to the tables it read and wrote - the system tables catalogued in [creating-system-data-layer](creating-system-data-layer/README.md) and the product tables catalogued in [setting-up-the-data-hub.ipynb](setting-up-the-data-hub.ipynb).

That is why this workbook runs **after** `setting-up-the-data-hub.ipynb`.  The cataloguer matches an event's tables to the tables Egeria has already catalogued.  If the events arrive first, it creates separate `DataSet` elements for the tables instead, and the ongoing update subscriptions set up in [digital-product-subscriptions.md](digital-product-subscriptions.md) - which watch the catalogued product tables - are never triggered.

Two things to know before running it:

* **Airflow must be sending its events to Egeria.**  From `compose-configs/optional-associated-runtimes/airflow-marquez`, run `./switch-lineage.sh egeria` (or `both`, to send them to Marquez as well).  The choice is kept in `openlineage.env`, so it only needs doing once.
* **The listener starts reading at the end of the topic.**  Only the events published after it is connected are catalogued; the ones already on the topic are not replayed.

The workbook can be run again safely: if the topic is already a catalog target of the listener, it does not add another one.

----

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = "peterprofile"
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

from IPython.display import display, Markdown
from pyegeria import load_mermaid, render_mermaid
load_mermaid()

from datetime import datetime
import time

# The classification explorer client is used to look up elements and their relationships
from pyegeria import ClassificationExplorer
classification_explorer = ClassificationExplorer(view_server, url, user_id, user_pwd)

# The automated curation client is for running the governance action process
from pyegeria import AutomatedCuration
automated_curation = AutomatedCuration(view_server, url, user_id, user_pwd)

# The governance officer client is for displaying governance action processes
from pyegeria import GovernanceOfficer
governance_officer = GovernanceOfficer(view_server, url, user_id, user_pwd)

# The runtime manager client is for controlling Egeria's servers
from pyegeria import RuntimeManager
runtime_manager = RuntimeManager(view_server, url, user_id, user_pwd, timeout=300)

def refresh_tokens():
    """Create a new bearer token and share it between the clients."""
    token = classification_explorer.create_egeria_bearer_token()
    automated_curation.set_bearer_token(token)
    governance_officer.set_bearer_token(token)
    runtime_manager.set_bearer_token(token)

refresh_tokens()

----

## Is the listener already connected?

The governance action process used below creates a `Topic` asset for `openlineage.events` and adds it as a catalog target of the `OpenLineageKafkaListener`.  Each run adds another catalog target, so the workbook first checks whether that has already been done.

----

In [ ]:
# Check whether the topic is already a catalog target of the listener

refresh_tokens()

OPEN_LINEAGE_KAFKA_LISTENER_GUID = "980b989c-de78-4e6a-a58d-51049d7381bf"   # OpenLineageKafkaListener integration connector
OPEN_LINEAGE_KAFKA_LISTENER_NAME = "OpenLineageKafkaListener"
TOPIC_QUALIFIED_NAME = "Topic::Kafka-Broker.openlineage.events::inOut"

def guid_of(qualified_name):
    """Return the GUID of the element with this qualified name, or None."""
    result = classification_explorer.get_element_guid_by_unique_name(qualified_name)
    return result if isinstance(result, str) and result != "No elements found" else None

def collect_guids(obj, found=None):
    """Walk a pyegeria response and gather every 'guid' value in it."""
    if found is None: found = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'guid' and isinstance(v, str): found.add(v)
            else: collect_guids(v, found)
    elif isinstance(obj, list):
        for v in obj: collect_guids(v, found)
    return found

def listener_is_connected():
    """True if the openlineage.events topic is a catalog target of the OpenLineageKafkaListener."""
    topic_guid = guid_of(TOPIC_QUALIFIED_NAME)
    if topic_guid is None:
        return False
    # CatalogTarget: end 1 is the integration connector, end 2 is the element being catalogued
    related = classification_explorer.get_related_elements(topic_guid, "CatalogTarget", start_at_end=2)
    return isinstance(related, list) and OPEN_LINEAGE_KAFKA_LISTENER_GUID in collect_guids(related)

already_connected = listener_is_connected()
if already_connected:
    print(f"{TOPIC_QUALIFIED_NAME} is already a catalog target of {OPEN_LINEAGE_KAFKA_LISTENER_NAME} - nothing to register.")
else:
    print(f"{OPEN_LINEAGE_KAFKA_LISTENER_NAME} is not listening to openlineage.events yet.")

# Retrieve the unique id (GUID) for the integration daemon, used to refresh the listener.  The daemon may have been
# catalogued under more than one platform URL, so prefer the element whose qualified name uses this workbook's URL.
integration_daemon_guid = None
servers = runtime_manager.get_servers_by_name(filter_string="qs-integration-daemon")
if servers and not isinstance(servers, str):
    for server in servers:
        if not server:
            continue
        server_qn = server.get("properties", {}).get("qualifiedName", "")
        if integration_daemon_guid is None or url in server_qn:
            integration_daemon_guid = server["elementHeader"]["guid"]
print(f"Integration daemon: {integration_daemon_guid}")

----

## Are the tables catalogued?

The lineage is only useful if each event's tables match the tables Egeria has already catalogued: the system tables from [creating-system-data-layer](creating-system-data-layer/README.md) and the product tables in `coco_data_hub` from [setting-up-the-data-hub.ipynb](setting-up-the-data-hub.ipynb).  Both are catalogued by the `JDBCDatabaseCataloguer`, which carries on in the background after those workbooks finish - for the system tables, around ten minutes.  If the listener is connected before it has caught up, the events about the tables it has not reached yet create separate `DataSet` elements.

So before connecting the listener, the cell below waits until every table is catalogued: each system schema has the number of tables listed in `creating-system-data-layer/data/system-databases.csv`, and every product table that the `coco_feed_*` DAGs write to - taken from their extract queries in `coco-dags/coco-system-extracts` - is in `coco_data_hub`.  It checks every 30 seconds for up to 20 minutes.  If the tables are still not all there, the listener is not connected; run the workbook again later.

----

In [ ]:
# Wait until Egeria has catalogued the tables that the events will refer to

import csv
from pathlib import Path

POSTGRES_SERVER_NAME = "Coco PostgreSQL Server 1"
SYSTEM_DATABASES_CSV = Path("creating-system-data-layer/data/system-databases.csv")
COCO_DAGS_DIRS       = [Path("/home/jovyan/coco-dags"),                  # as mounted in JupyterLab
                        Path("../../exchange-quickstart/coco-dags")]     # in the repository
WAIT_MINUTES         = 20

# The expected table count for each schema, keyed by the qualified name prefix of its tables
expected_tables = {}
with open(SYSTEM_DATABASES_CSV, newline='', encoding='utf-8') as fh:
    for row in csv.DictReader(fh):
        prefix = f"PostgreSQL Relational Database Schema::{POSTGRES_SERVER_NAME}::{row['database_name']}.{row['schema_name']}::"
        expected_tables[prefix] = int(row['table_count'])
system_schema_count = len(expected_tables)

coco_dags_dir = next((d for d in COCO_DAGS_DIRS if (d / "coco-system-extracts").is_dir()), None)
if coco_dags_dir is None:
    print("The coco-dags extract queries are not visible from here, so only the system tables are checked.")
else:
    product_tables = {}
    for extract in (coco_dags_dir / "coco-system-extracts").glob("*/*/*.sql"):
        product_schema, _, product_table = extract.stem.partition(".")
        product_tables.setdefault(product_schema, set()).add(product_table)
    for product_schema, tables in product_tables.items():
        prefix = f"PostgreSQL Relational Database::{POSTGRES_SERVER_NAME}::coco_data_hub::{product_schema}::"
        expected_tables[prefix] = len(tables)

def catalogued_table_count(prefix):
    """The number of RelationalTables whose qualified name starts with this prefix."""
    result = classification_explorer.find_elements_by_property_value(property_value=prefix,
                                                                     property_names=["qualifiedName"],
                                                                     metadata_element_type_name="RelationalTable",
                                                                     starts_with=True,
                                                                     graph_query_depth=0)
    return len(result) if isinstance(result, list) else 0

print(f"Checking {sum(expected_tables.values())} tables: {system_schema_count} system schemas "
      f"and {len(expected_tables) - system_schema_count} product schemas in coco_data_hub")

tables_catalogued = already_connected
if already_connected:
    print("Skipped - the listener is already connected.")
else:
    waiting  = dict(expected_tables)       # only the schemas still short are checked again
    found    = {}
    deadline = time.time() + WAIT_MINUTES * 60
    while True:
        refresh_tokens()
        for prefix in list(waiting):
            found[prefix] = catalogued_table_count(prefix)
            if found[prefix] >= waiting[prefix]:
                del waiting[prefix]
        if not waiting or time.time() > deadline:
            break
        print(f"{datetime.now():%H:%M:%S}  {len(waiting)} of {len(expected_tables)} schemas are still being catalogued")
        time.sleep(30)

    tables_catalogued = not waiting
    if tables_catalogued:
        print("All of the tables are catalogued - the listener can be connected.")
    else:
        lines = ["| Schema | Tables found | Tables expected |", "|---|---|---|"]
        for prefix, count in sorted(waiting.items()):
            lines.append(f"| `{prefix.split('::', 2)[-1].rstrip(':')}` | {found.get(prefix, 0)} | {count} |")
        display(Markdown("\n".join(lines)))
        print(f"These schemas are still not fully catalogued after {WAIT_MINUTES} minutes, so the listener is not "
              "connected.  Check the JDBCDatabaseCataloguer, then run this workbook again.")

----

## Registering the topic with the listener

Egeria has a predefined [governance action process](https://egeria-project.org/concepts/governance-action-process/) for this.  It creates the `Topic` asset that describes `openlineage.events` on the Apache Kafka broker in `shared-infra`, and then adds it as a catalog target of the `OpenLineageKafkaListener`.  The process is displayed below and then run - unless the checks above found that it has already been done, or that the tables are not all catalogued yet.

----

In [ ]:
# Display the process and run it

refresh_tokens()

harvest_open_lineage_events = "HarvestOpenLineageEvents::CreateAsCatalogTargetGovernanceActionProcess"

process_guid = classification_explorer.get_element_guid_by_unique_name(harvest_open_lineage_events)
process_graph = governance_officer.get_governance_action_process_graph(process_guid, output_format="JSON")
render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

instance_guid = None
if already_connected:
    print("Skipped - the listener is already connected.")
elif not tables_catalogued:
    print("Not started - the tables are not all catalogued yet.  Run this workbook again once they are.")
else:
    print("Starting the " + harvest_open_lineage_events + " process")

    request_parameters = {
        "serverName" : "Kafka-Broker",
        "hostIdentifier" : "host.docker.internal",
        "portNumber" : "9192",
        "fullTopicName" : "openlineage.events",
        "shortTopicName" : "openlineage.events",
        "eventDirection" : "inOut",
        "versionIdentifier" : "1.0",
        "description" : "The topic that the OpenLineage proxy publishes Apache Airflow's OpenLineage events to."
    }

    instance_guid = automated_curation.initiate_gov_action_process(action_type_qualified_name=harvest_open_lineage_events,
                                                                   request_source_guids=None,
                                                                   action_targets=None,
                                                                   start_time=datetime.now(),
                                                                   request_parameters=request_parameters,
                                                                   orig_service_name=None,
                                                                   orig_engine_name=None)

----

The process runs in the background.  The cell below waits until the topic has been added to the listener's catalog targets - usually a few seconds - and then shows the path the process took.

----

In [ ]:
# Wait for the process to connect the listener, then show the execution path

refresh_tokens()

if instance_guid is not None:
    for attempt in range(36):                     # up to 3 minutes
        if listener_is_connected():
            break
        time.sleep(5)

    if listener_is_connected():
        print(f"{TOPIC_QUALIFIED_NAME} is now a catalog target of {OPEN_LINEAGE_KAFKA_LISTENER_NAME}.")
    else:
        print("The listener is not connected yet - check the process below for a FAILED step.")

    process_graph = governance_officer.get_governance_action_process_graph(instance_guid)
    render_mermaid(process_graph.get("governanceActionProcessMermaidGraph"))

----

## Starting the listener

The listener is refreshed so that it opens its connection to the topic now rather than at its next scheduled refresh.  From this point, each event that Airflow publishes is catalogued.

----

In [ ]:
refresh_tokens()

# Refresh the listener
# An asynchronous call is used since this will return once the refresh is complete - which can result in a timeout on a normal synchronous request.

try:
    print("Refreshing: " + OPEN_LINEAGE_KAFKA_LISTENER_NAME)
    await runtime_manager._async_refresh_integration_connector(server_guid=integration_daemon_guid,
                                                               connector_name=OPEN_LINEAGE_KAFKA_LISTENER_NAME)
except Exception:
    print("Timed out - the listener carries on starting in the background.")

print("Completed.")

In [ ]:
# Report on the listener.
# The daemon is looked up by its GUID rather than by name, since the name can match more than one element.

refresh_tokens()

daemon_report = runtime_manager.get_server_report(server_guid=integration_daemon_guid)
connector_reports = daemon_report.get("integrationConnectorReports", []) if isinstance(daemon_report, dict) else []
listener_report = next((r for r in connector_reports if r.get("connectorName") == OPEN_LINEAGE_KAFKA_LISTENER_NAME), None)

if listener_report is None:
    print(f"{OPEN_LINEAGE_KAFKA_LISTENER_NAME} is not in the report for integration daemon {integration_daemon_guid}")
else:
    lines = ["| Connector | Status | Last refresh | Exception |", "|---|---|---|---|",
             f"| {OPEN_LINEAGE_KAFKA_LISTENER_NAME} | {listener_report.get('connectorStatus', '---')} | "
             f"{listener_report.get('lastRefreshTime', '---')} | {listener_report.get('failingExceptionMessage') or ''} |"]
    display(Markdown("\n".join(lines)))

----

## Seeing the lineage arrive

Once Airflow has run a few tasks, their lineage appears in Egeria.  Each task of a DAG is catalogued as a `DeployedSoftwareComponent` whose qualified name starts with `DeployedSoftwareComponent::egeria-workspace::` followed by the DAG and task names.  The cell below counts them for each DAG; run it again after a few minutes to watch the numbers grow.

----

In [ ]:
# Count the Airflow tasks catalogued from the OpenLineage events, by DAG

refresh_tokens()

from collections import Counter

tasks = classification_explorer.find_elements_by_property_value(property_value="DeployedSoftwareComponent::egeria-workspace::",
                                                                property_names=["qualifiedName"],
                                                                metadata_element_type_name="DeployedSoftwareComponent",
                                                                starts_with=True,
                                                                graph_query_depth=0)
by_dag = Counter()
if isinstance(tasks, list):
    for task in tasks:
        name = task.get('properties', {}).get('qualifiedName', '').split("::", 2)[-1]
        by_dag[name.split(".", 1)[0]] += 1

if by_dag:
    lines = ["| DAG | Tasks catalogued |", "|---|---|"] + [f"| {dag} | {count} |" for dag, count in sorted(by_dag.items())]
    display(Markdown("\n".join(lines)))
else:
    print("No Airflow tasks catalogued yet - check that Airflow is running and sending its events to Egeria.")

----

## In summary

Egeria now receives the OpenLineage events that Airflow publishes, and records the lineage of every DAG task against the tables catalogued in the earlier steps.  The next step, [digital-product-subscriptions.md](digital-product-subscriptions.md), sets up the subscriptions that deliver the digital products to the systems that depend on them; its ongoing update subscriptions are triggered by this lineage.

----